In [1]:
!pip install sentence-transformers -q

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import csv

In [3]:
from google.colab import files
uploaded = files.upload()

Saving animal-fun-facts-dataset.csv to animal-fun-facts-dataset.csv


In [4]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts, show_progress_bar=False)
        new_embeddings = np.array(new_embeddings)

        if self.embeddings is None:
            self.embeddings = new_embeddings
            self.documents = list(documents)
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
            self.documents.extend(documents)

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        query_embedding = self.model.encode([query])
        query_embedding = np.array(query_embedding)

        dot_products = np.dot(self.embeddings, query_embedding.T).flatten()
        doc_norms = np.linalg.norm(self.embeddings, axis=1)
        query_norm = np.linalg.norm(query_embedding)
        similarities = dot_products / (doc_norms * query_norm + 1e-10)

        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append(SearchResult(
                score=float(similarities[idx]),
                document=self.documents[idx]
            ))
        return results

In [5]:
def load_animal_facts(filepath: str) -> list[Document]:
    documents = []
    with open(filepath, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            text = row["text"].strip()
            if not text:
                continue
            metadata = {
                "animal_name": row["animal_name"],
                "source": row["source"],
                "media_link": row["media_link"],
                "wikipedia_link": row["wikipedia_link"],
            }
            documents.append(Document(text=text, metadata=metadata))
    return documents

animal_docs = load_animal_facts("animal-fun-facts-dataset.csv")
print(f"Total documents loaded: {len(animal_docs)}")
print(f"Sample: {animal_docs[0].text[:100]}...")
print(f"Metadata: {animal_docs[0].metadata}")

Total documents loaded: 7731
Sample: Aardvarks are sometimes called "ant bears", "earth pigs",
and "cape anteaters"...
Metadata: {'animal_name': 'aardvark', 'source': 'https://www.animalfactsencyclopedia.com/Aardvark-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Aardvark'}


In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

subset = animal_docs[:800]
store = VectorStore(model)
store.add_documents(subset)
print(f"VectorStore created with {len(store.documents)} documents")
print(f"Embedding matrix shape: {store.embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStore created with 800 documents
Embedding matrix shape: (800, 384)


In [7]:
def show_results(query: str, results: list[SearchResult]):
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")
    for i, r in enumerate(results, 1):
        print(f"\n--- Result {i} (score: {r.score:.4f}) ---")
        print(f"Text: {r.document.text}")
        print(f"Metadata: {r.document.metadata}")

In [8]:
queries = [
    "animals that live in the ocean",
    "fastest animals in the world",
    "animals with unusual sleeping habits",
    "poisonous or venomous creatures",
    "animals that can fly",
]

for q in queries:
    results = store.search(q, top_k=3)
    show_results(q, results)


Query: animals that live in the ocean

--- Result 1 (score: 0.5888) ---
Text: Walruses live in the oceans around the North Pole
Metadata: {'animal_name': 'walrus', 'source': 'https://www.animalfactsencyclopedia.com/Walrus-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/Walrus'}

--- Result 2 (score: 0.5837) ---
Text: Hard ocean floors are their home.
Unlike other fish who swim up and down in the water column, these fish prefer to make their homes on the rocky floor of the ocean. This provides them with ample access to the prey that make up their diet.
Metadata: {'animal_name': 'atlantic wolffish', 'source': 'https://factanimal.com/atlantic-wolffish/', 'media_link': '', 'wikipedia_link': '/wiki/Atlantic_wolffish'}

--- Result 3 (score: 0.5784) ---
Text: They have many different names.
Atlantic wolffish are also called seawolves, ocean catfish, devil fish, sea cats, and woofs. Their Pacific Ocean relative (the wolffish, Anarrhichthys ocellatus) is known as the wolf eel.
Metadata

In [9]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.model = embedding_model
        self.documents: list[Document] = []
        self.embeddings: np.ndarray | None = None

    def add_documents(self, documents: list[Document]):
        texts = [doc.text for doc in documents]
        new_embeddings = self.model.encode(texts, show_progress_bar=False)
        new_embeddings = np.array(new_embeddings)

        if self.embeddings is None:
            self.embeddings = new_embeddings
            self.documents = list(documents)
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
            self.documents.extend(documents)

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        query_embedding = self.model.encode([query])
        query_embedding = np.array(query_embedding)

        dot_products = np.dot(self.embeddings, query_embedding.T).flatten()
        doc_norms = np.linalg.norm(self.embeddings, axis=1)
        query_norm = np.linalg.norm(query_embedding)
        similarities = dot_products / (doc_norms * query_norm + 1e-10)

        if metadata_filter:
            valid_indices = []
            for i, doc in enumerate(self.documents):
                match = all(
                    doc.metadata.get(key) == value
                    for key, value in metadata_filter.items()
                )
                if match:
                    valid_indices.append(i)
            valid_indices = np.array(valid_indices)
            if len(valid_indices) == 0:
                return []
            filtered_sims = similarities[valid_indices]
            top_local = np.argsort(filtered_sims)[::-1][:top_k]
            top_indices = valid_indices[top_local]
        else:
            top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append(SearchResult(
                score=float(similarities[idx]),
                document=self.documents[idx]
            ))
        return results

In [10]:
product_reviews = [
    #headphones
    Document("These wireless headphones have incredible noise cancellation that blocks out everything around you. Perfect for flights and commutes.", {"category": "headphones", "brand": "Sony", "sentiment": "positive"}),
    Document("The bass response on these headphones is muddy and the highs are tinny. Not worth the premium price tag at all.", {"category": "headphones", "brand": "Beats", "sentiment": "negative"}),
    Document("Comfortable over-ear design with excellent battery life lasting over 30 hours on a single charge.", {"category": "headphones", "brand": "Sony", "sentiment": "positive"}),
    Document("The Bluetooth connection drops constantly when I walk more than 10 feet from my phone. Very frustrating experience.", {"category": "headphones", "brand": "Beats", "sentiment": "negative"}),
    Document("Studio-quality sound with balanced mids and crystal clear highs. The spatial audio feature is mind-blowing.", {"category": "headphones", "brand": "Apple", "sentiment": "positive"}),
    Document("The ear cushions started peeling after just three months of normal use. Build quality is disappointing.", {"category": "headphones", "brand": "Beats", "sentiment": "negative"}),
    Document("Lightweight design makes these perfect for long listening sessions. The fold-up mechanism is convenient for travel.", {"category": "headphones", "brand": "Bose", "sentiment": "positive"}),
    Document("Active noise cancellation works well on low frequencies but lets through a lot of high-pitched sounds like voices.", {"category": "headphones", "brand": "Bose", "sentiment": "neutral"}),
    # smartphones
    Document("The camera on this phone is absolutely stunning. Night mode produces photos that rival dedicated cameras.", {"category": "smartphone", "brand": "Apple", "sentiment": "positive"}),
    Document("Battery barely lasts half a day with normal usage. I have to carry a power bank everywhere now.", {"category": "smartphone", "brand": "Samsung", "sentiment": "negative"}),
    Document("The 120Hz display is buttery smooth and makes scrolling and animations feel incredibly responsive.", {"category": "smartphone", "brand": "Samsung", "sentiment": "positive"}),
    Document("This phone heats up significantly during video calls and gaming. Thermal management needs improvement.", {"category": "smartphone", "brand": "Google", "sentiment": "negative"}),
    Document("AI-powered photo editing features are a game changer. Magic eraser removes unwanted objects seamlessly.", {"category": "smartphone", "brand": "Google", "sentiment": "positive"}),
    Document("The phone is too large for one-handed use and the weight makes it uncomfortable during long calls.", {"category": "smartphone", "brand": "Samsung", "sentiment": "negative"}),
    Document("Fast charging goes from 0 to 80 percent in under 30 minutes. Wireless charging also works flawlessly.", {"category": "smartphone", "brand": "Apple", "sentiment": "positive"}),
    #laptops
    Document("The M-series chip delivers incredible performance while keeping the laptop completely silent during heavy tasks.", {"category": "laptop", "brand": "Apple", "sentiment": "positive"}),
    Document("The keyboard feels mushy and lacks the tactile feedback I need for comfortable all-day typing sessions.", {"category": "laptop", "brand": "Dell", "sentiment": "negative"}),
    Document("This laptop handles video editing and 3D rendering without breaking a sweat. Export times are impressively fast.", {"category": "laptop", "brand": "Apple", "sentiment": "positive"}),
    Document("The trackpad is too small and frequently registers accidental palm touches while typing documents.", {"category": "laptop", "brand": "Lenovo", "sentiment": "negative"}),
    Document("Fan noise under load is extremely loud, making it impossible to use in quiet environments like libraries.", {"category": "laptop", "brand": "Dell", "sentiment": "negative"}),
    Document("The OLED display has perfect blacks and vibrant colors that make creative work an absolute pleasure.", {"category": "laptop", "brand": "Lenovo", "sentiment": "positive"}),
    Document("Excellent build quality with a solid aluminum chassis that feels premium without being too heavy to carry.", {"category": "laptop", "brand": "Apple", "sentiment": "positive"}),
    #speakers
    Document("Room-filling sound from such a compact speaker is remarkable. The bass is deep and rich for its size.", {"category": "speaker", "brand": "Sonos", "sentiment": "positive"}),
    Document("The voice assistant integration is clunky and often misunderstands basic commands for music playback.", {"category": "speaker", "brand": "Amazon", "sentiment": "negative"}),
    Document("Multi-room audio setup was seamless and the synchronization between speakers is perfect with zero delay.", {"category": "speaker", "brand": "Sonos", "sentiment": "positive"}),
    Document("Waterproof design and rugged build make this the ideal speaker for pool parties and outdoor adventures.", {"category": "speaker", "brand": "JBL", "sentiment": "positive"}),
    Document("The app required for setup is buggy and crashes frequently. Took over an hour to get the speaker connected.", {"category": "speaker", "brand": "Amazon", "sentiment": "negative"}),
    Document("Portable speaker with surprisingly powerful output. The 360-degree sound fills a room evenly from any position.", {"category": "speaker", "brand": "JBL", "sentiment": "positive"}),
    #smartwatches
    Document("Health tracking is comprehensive with heart rate, blood oxygen, sleep stages, and stress monitoring all accurate.", {"category": "smartwatch", "brand": "Apple", "sentiment": "positive"}),
    Document("The screen is nearly impossible to read in direct sunlight even at maximum brightness setting.", {"category": "smartwatch", "brand": "Fitbit", "sentiment": "negative"}),
    Document("GPS tracking for runs is very accurate and the workout detection automatically identifies exercise types.", {"category": "smartwatch", "brand": "Garmin", "sentiment": "positive"}),
    Document("The battery only lasts about 18 hours which means charging every single night without fail.", {"category": "smartwatch", "brand": "Apple", "sentiment": "negative"}),
    Document("Rugged design with sapphire crystal glass and titanium case that survives drops and scratches with ease.", {"category": "smartwatch", "brand": "Garmin", "sentiment": "positive"}),
]

print(f"Product reviews dataset: {len(product_reviews)} documents")
categories = set(d.metadata["category"] for d in product_reviews)
brands = set(d.metadata["brand"] for d in product_reviews)
print(f"Categories: {categories}")
print(f"Brands: {brands}")

Product reviews dataset: 33 documents
Categories: {'speaker', 'laptop', 'smartphone', 'headphones', 'smartwatch'}
Brands: {'Google', 'Apple', 'Sony', 'Samsung', 'Amazon', 'Beats', 'Fitbit', 'Lenovo', 'JBL', 'Sonos', 'Garmin', 'Bose', 'Dell'}


In [11]:
filtered_store = FilteredVectorStore(model)
filtered_store.add_documents(product_reviews)
print(f"FilteredVectorStore created with {len(filtered_store.documents)} documents")

FilteredVectorStore created with 33 documents


In [12]:
def show_filtered_results(query, results, metadata_filter=None):
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    if metadata_filter:
        print(f"Filter: {metadata_filter}")
    print(f"{'='*80}")
    if not results:
        print("No results found matching the filter criteria.")
    for i, r in enumerate(results, 1):
        print(f"\n--- Result {i} (score: {r.score:.4f}) ---")
        print(f"Text: {r.document.text}")
        print(f"Metadata: {r.document.metadata}")

In [13]:
# Query 1: Sound quality, headphones only
q1 = "great sound quality and audio"
f1 = {"category": "headphones"}
show_filtered_results(q1, filtered_store.search(q1, top_k=3, metadata_filter=f1), f1)

# Query 2: Samsung negative reviews only
q2 = "problems and issues with the device"
f2 = {"brand": "Samsung", "sentiment": "negative"}
show_filtered_results(q2, filtered_store.search(q2, top_k=3, metadata_filter=f2), f2)

# Query 3: Battery life, positive reviews only
q3 = "battery life and charging"
f3 = {"sentiment": "positive"}
show_filtered_results(q3, filtered_store.search(q3, top_k=3, metadata_filter=f3), f3)

# Query 4: Apple laptops specifically
q4 = "performance and speed for professional work"
f4 = {"category": "laptop", "brand": "Apple"}
show_filtered_results(q4, filtered_store.search(q4, top_k=3, metadata_filter=f4), f4)

# Query 5: JBL products
q5 = "portable durable outdoor use"
f5 = {"brand": "JBL"}
show_filtered_results(q5, filtered_store.search(q5, top_k=3, metadata_filter=f5), f5)


Query: great sound quality and audio
Filter: {'category': 'headphones'}

--- Result 1 (score: 0.6043) ---
Text: Studio-quality sound with balanced mids and crystal clear highs. The spatial audio feature is mind-blowing.
Metadata: {'category': 'headphones', 'brand': 'Apple', 'sentiment': 'positive'}

--- Result 2 (score: 0.4228) ---
Text: The bass response on these headphones is muddy and the highs are tinny. Not worth the premium price tag at all.
Metadata: {'category': 'headphones', 'brand': 'Beats', 'sentiment': 'negative'}

--- Result 3 (score: 0.3353) ---
Text: Active noise cancellation works well on low frequencies but lets through a lot of high-pitched sounds like voices.
Metadata: {'category': 'headphones', 'brand': 'Bose', 'sentiment': 'neutral'}

Query: problems and issues with the device
Filter: {'brand': 'Samsung', 'sentiment': 'negative'}

--- Result 1 (score: 0.1157) ---
Text: The phone is too large for one-handed use and the weight makes it uncomfortable during long call

Extra

In [14]:
!pip install fastapi uvicorn nest-asyncio -q

In [15]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, field_validator
import uvicorn, nest_asyncio, threading, uuid, time
import requests

In [16]:
nest_asyncio.apply()


REQUIRED_METADATA_KEYS = {"category", "brand", "sentiment"}

class DocumentInput(BaseModel):
    text: str
    metadata: dict[str, str]

    @field_validator("metadata")
    @classmethod
    def validate_metadata(cls, v):
        if set(v.keys()) != REQUIRED_METADATA_KEYS:
            raise ValueError(
                f"Metadata must contain exactly these keys: {REQUIRED_METADATA_KEYS}. "
                f"Got: {set(v.keys())}"
            )
        return v

class SearchInput(BaseModel):
    query: str
    top_k: int = 5
    metadata_filter: dict[str, str] | None = None


def chunk_text(text: str, max_length: int = 500, chunk_size: int = 400) -> list[str]:
    if len(text) <= max_length:
        return [text]
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])
    return chunks


app = FastAPI(title="FilteredVectorStore API")

original_documents: dict[str, DocumentInput] = {}
api_store = FilteredVectorStore(model)

@app.post("/documents")
def create_document(doc: DocumentInput):
    doc_id = str(uuid.uuid4())
    original_documents[doc_id] = doc

    chunks = chunk_text(doc.text)
    chunk_docs = []
    for i, chunk in enumerate(chunks):
        meta = {**doc.metadata, "original_doc_id": doc_id, "chunk_index": str(i)}
        chunk_docs.append(Document(text=chunk, metadata=meta))

    api_store.add_documents(chunk_docs)

    return {
        "id": doc_id,
        "chunks_created": len(chunks),
        "message": f"Document stored with {len(chunks)} chunk(s)"
    }

@app.get("/documents/{doc_id}")
def get_document(doc_id: str):
    if doc_id not in original_documents:
        raise HTTPException(status_code=404, detail="Document not found")
    doc = original_documents[doc_id]
    return {
        "id": doc_id,
        "text": doc.text,
        "metadata": doc.metadata
    }

@app.post("/documents/search")
def search_documents(search: SearchInput):
    results = api_store.search(
        query=search.query,
        top_k=search.top_k,
        metadata_filter=search.metadata_filter
    )
    return [
        {
            "score": round(r.score * 100, 2),
            "text": r.document.text,
            "metadata": r.document.metadata
        }
        for r in results
    ]


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

print("API running on http://localhost:8000")
print("Docs: http://localhost:8000/docs")

API running on http://localhost:8000
Docs: http://localhost:8000/docs


In [17]:
BASE = "http://localhost:8000"

print("="*80)
print("POST /documents — Creating documents")
print("="*80)

test_docs = [
    {
        "text": "These over-ear headphones deliver exceptional clarity across all frequencies. The soundstage is wide and immersive, making them ideal for critical listening sessions. The memory foam ear cushions provide hours of comfortable use without any fatigue. Build quality is outstanding with premium materials throughout. The detachable cable system supports both balanced and unbalanced connections. Frequency response extends from 6Hz to 38kHz giving incredible detail in both sub-bass and treble regions. The included carrying case is hardshell and fits the headphones perfectly with room for accessories and cables.",
        "metadata": {"category": "headphones", "brand": "Sennheiser", "sentiment": "positive"}
    },
    {
        "text": "Compact smart speaker with room-filling sound.",
        "metadata": {"category": "speaker", "brand": "Sonos", "sentiment": "positive"}
    },
    {
        "text": "Battery drains in under 4 hours. Complete waste of money for a portable device.",
        "metadata": {"category": "speaker", "brand": "JBL", "sentiment": "negative"}
    },
]

doc_ids = []
for doc in test_docs:
    r = requests.post(f"{BASE}/documents", json=doc)
    result = r.json()
    print(f"\n{result}")
    doc_ids.append(result["id"])

print(f"\n{'='*80}")
print("POST /documents — Invalid metadata (should return 422)")
print("="*80)

bad_doc = {"text": "Some text", "metadata": {"wrong_key": "value"}}
r = requests.post(f"{BASE}/documents", json=bad_doc)
print(f"Status: {r.status_code}")
print(f"Response: {r.json()['detail'][0]['msg']}")

print(f"\n{'='*80}")
print(f"GET /documents/{{id}} — Retrieve first document (chunked)")
print("="*80)

r = requests.get(f"{BASE}/documents/{doc_ids[0]}")
result = r.json()
print(f"\nID: {result['id']}")
print(f"Full text: {result['text'][:100]}...")
print(f"Metadata: {result['metadata']}")

print(f"\n{'='*80}")
print("POST /documents/search — 'comfortable listening'")
print("="*80)

r = requests.post(f"{BASE}/documents/search", json={
    "query": "comfortable listening",
    "top_k": 3
})
for res in r.json():
    print(f"\n  Score: {res['score']}%")
    print(f"  Text: {res['text']}")
    print(f"  Metadata: {res['metadata']}")

print(f"\n{'='*80}")
print("POST /documents/search — 'battery' filtered by sentiment=negative")
print("="*80)

r = requests.post(f"{BASE}/documents/search", json={
    "query": "battery life",
    "top_k": 3,
    "metadata_filter": {"sentiment": "negative"}
})
for res in r.json():
    print(f"\n  Score: {res['score']}%")
    print(f"  Text: {res['text']}")
    print(f"  Metadata: {res['metadata']}")

POST /documents — Creating documents

{'id': 'cccc49b0-cf80-48d0-9a52-a6b091d3d944', 'chunks_created': 2, 'message': 'Document stored with 2 chunk(s)'}

{'id': '1bfc9987-ae37-4cc9-b64b-a09782e7f91f', 'chunks_created': 1, 'message': 'Document stored with 1 chunk(s)'}

{'id': '8d66dc4c-3dd7-4945-a12b-389cb8ab635b', 'chunks_created': 1, 'message': 'Document stored with 1 chunk(s)'}

POST /documents — Invalid metadata (should return 422)
Status: 422
Response: Value error, Metadata must contain exactly these keys: {'sentiment', 'brand', 'category'}. Got: {'wrong_key'}

GET /documents/{id} — Retrieve first document (chunked)

ID: cccc49b0-cf80-48d0-9a52-a6b091d3d944
Full text: These over-ear headphones deliver exceptional clarity across all frequencies. The soundstage is wide...
Metadata: {'category': 'headphones', 'brand': 'Sennheiser', 'sentiment': 'positive'}

POST /documents/search — 'comfortable listening'

  Score: 38.17%
  Text: These over-ear headphones deliver exceptional clarity ac

This activity allowed me to understand how vector stores work internally, rather than relying on libraries that abstract away all the logic.

Converting text to numerical vectors using SentenceTransformer captures the meaning of sentences, not just keywords. Queries like "animals that live in the ocean" find results about marine creatures even when they don't contain that exact phrase. Implementing it manually with numpy helped me understand the geometry behind semantic search. Combining semantic search with filters is essential for real-world applications. A user might search for "great audio quality" but only among products from a specific brand or category. Exposing the vector store as an API showed me how these systems work in production. Chunking is necessary because embedding models have input limits, and splitting long documents enables a more precise level search while preserving a reference back to the original document. With more than 7,000 documents, embedding time is significant. This explains why production systems use approximate nearest neighbor indexes and specialized vector databases.

The biggest obstacle wasn't the implementation itself but the saving through GitHub. Google Colab injects Jupyter widget metadata (`metadata.widgets`) into the file when libraries like SentenceTransformer display progress bars during encoding. GitHub's notebook renderer cannot handle this malformed metadata and displays an "Invalid Notebook" error instead of rendering the file. The notebook can be opened directly in Colab from the repository without issues.

Generative AI was used to correct grammar and solve errors.